# Deepfake Face Detection: training on a Kaggle GPU

Trains an EfficientNet-B0 real/fake classifier on **140k Real and Fake Faces** and shows a live
progress panel (images trained, images left, speed, ETA).

**Before you run it**
1. Right panel, *Add Input*: search for **140k Real and Fake Faces** (by xhlulu) and add it.
2. Right panel, *Session options*: set **Accelerator** to a GPU (T4 or P100).
3. Same panel: turn **Internet** on, so the pretrained weights can download (needs a phone-verified account).
4. *Run All*, and keep this tab open to watch the panel.

When it finishes, download `best.pt` from the *Output* tab and save it as `checkpoints/best.pt` in your project.

In [ ]:
import os
import time
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ---- settings -------------------------------------------------------------
EPOCHS = int(os.environ.get("EPOCHS", 5))
BATCH_SIZE = 128
LR = 1e-4
IMAGE_SIZE = 224
TRAIN_SUBSET = int(os.environ.get("TRAIN_SUBSET", 0)) or None   # e.g. 30000 for a faster run
VAL_SUBSET = int(os.environ.get("VAL_SUBSET", 0)) or None
TEST_SUBSET = int(os.environ.get("TEST_SUBSET", 0)) or None

ON_KAGGLE = Path("/kaggle/input").exists()
DATA_DIR = "/kaggle/input" if ON_KAGGLE else "data"
OUT_DIR = Path("/kaggle/working" if ON_KAGGLE else "outputs")
OUT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Device:", torch.cuda.get_device_name(0) if use_amp else "CPU (no GPU found: turn on an Accelerator)")

In [ ]:
def find_root(base):
    """Find the folder holding train/valid/test, wherever the dataset was mounted."""
    for folder, subdirs, _ in os.walk(base):
        if {"train", "valid", "test"} <= set(subdirs):
            return Path(folder)
    raise FileNotFoundError("Dataset not found. Use 'Add Input' to add xhlulu/140k-real-and-fake-faces.")


MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def make_loader(root, split, subset=None):
    train = split == "train"
    steps = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if train:
        steps += [transforms.RandomHorizontalFlip(), transforms.ColorJitter(brightness=0.1, contrast=0.1)]
    steps += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    dataset = datasets.ImageFolder(root / split, transform=transforms.Compose(steps))

    if subset and subset < len(dataset):
        generator = torch.Generator().manual_seed(0)
        dataset = Subset(dataset, torch.randperm(len(dataset), generator=generator)[:subset].tolist())

    workers = min(4, os.cpu_count() or 1)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=train,
        num_workers=workers,
        pin_memory=use_amp,
        persistent_workers=workers > 0,
        prefetch_factor=4 if workers > 0 else None,
    )


root = find_root(DATA_DIR)
train_loader = make_loader(root, "train", TRAIN_SUBSET)
val_loader = make_loader(root, "valid", VAL_SUBSET)
test_loader = make_loader(root, "test", TEST_SUBSET)
CLASS_NAMES = ["fake", "real"]  # folder order: fake=0, real=1
print(f"Data: {root}")
print(f"train {len(train_loader.dataset):,}  valid {len(val_loader.dataset):,}  test {len(test_loader.dataset):,}")

In [ ]:
def fmt_time(seconds):
    seconds = int(max(seconds, 0))
    h, rest = divmod(seconds, 3600)
    m, s = divmod(rest, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m}m {s:02d}s"


def bar(percent, color):
    return (
        '<div style="background:#8883;border-radius:6px;height:14px;overflow:hidden">'
        f'<div style="width:{percent:.1f}%;height:100%;background:{color}"></div></div>'
    )


def tile(label, value):
    return (
        '<div style="flex:1;min-width:130px;background:#8881;border-radius:8px;padding:8px 12px">'
        f'<div style="font-size:12px;opacity:.7">{label}</div>'
        f'<div style="font-size:18px;font-weight:600">{value}</div></div>'
    )


class Dashboard:
    """Live progress panel, redrawn in place at most once a second."""

    def __init__(self, epochs, train_n, val_n):
        self.epochs, self.train_n, self.val_n = epochs, train_n, val_n
        self.total = epochs * (train_n + val_n)   # every image processed, training and validation
        self.done = 0
        self.trained = 0
        self.epoch, self.phase = 1, "Starting"
        self.in_phase, self.phase_total = 0, 1
        self.loss = self.acc = None
        self.history = []
        self.t0 = time.time()
        self.last_draw = 0.0
        # Background (non-interactive) runs have no live panel, so also write a line to the log each minute.
        self.log_lines = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"
        self.last_log = 0.0
        self.handle = display(HTML(""), display_id=True)

    def update(self, force=False):
        now = time.time()
        if not force and now - self.last_draw < 1:
            return
        self.last_draw = now

        elapsed = now - self.t0
        speed = self.done / elapsed if self.done else 0
        eta = fmt_time((self.total - self.done) / speed) if speed else "..."
        train_total = self.epochs * self.train_n
        overall = 100 * self.done / self.total
        phase_pct = 100 * self.in_phase / self.phase_total
        if self.log_lines and now - self.last_log >= 60:
            self.last_log = now
            print(f"[{fmt_time(elapsed)}] epoch {min(self.epoch, self.epochs)}/{self.epochs} {self.phase.lower()}: "
                  f"trained {self.trained:,}/{train_total:,} images, {speed:,.0f} img/s, ETA ~{eta}", flush=True)
        pass_label = "Run complete" if self.phase == "Done" else f"This {self.phase.lower()} pass"
        live = "" if self.loss is None else f" &nbsp; loss {self.loss:.4f} &nbsp; acc {self.acc:.4f}"

        rows = "".join(
            f"<tr><td>{h['epoch']}</td><td>{h['train_loss']:.4f}</td><td>{h['train_acc']:.4f}</td>"
            f"<td>{h['val_loss']:.4f}</td><td>{h['val_acc']:.4f}</td><td>{fmt_time(h['seconds'])}</td></tr>"
            for h in self.history
        )
        table = (
            '<table style="margin-top:12px;border-collapse:collapse;font-size:13px;width:100%">'
            '<tr style="text-align:left;opacity:.7"><th>Epoch</th><th>Train loss</th><th>Train acc</th>'
            f"<th>Val loss</th><th>Val acc</th><th>Time</th></tr>{rows}</table>"
            if rows else ""
        )

        self.handle.update(HTML(
            '<div style="font-family:system-ui,sans-serif;max-width:760px">'
            f'<div style="font-size:16px;font-weight:600;margin-bottom:8px">Epoch {min(self.epoch, self.epochs)} of {self.epochs}'
            f" &nbsp;&middot;&nbsp; {self.phase}</div>"
            '<div style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:12px">'
            + tile("Images trained", f"{self.trained:,}")
            + tile("Left to train", f"{train_total - self.trained:,}")
            + tile("Speed", f"{speed:,.0f} img/s")
            + tile("Elapsed / ETA", f"{fmt_time(elapsed)} / ~{eta}")
            + "</div>"
            f'<div style="font-size:13px;margin-bottom:4px">Overall: {overall:.1f}% '
            f"({self.trained:,} of {train_total:,} training images seen)</div>"
            + bar(overall, "#2a7de1")
            + f'<div style="font-size:13px;margin:10px 0 4px">{pass_label}: {phase_pct:.1f}%{live}</div>'
            + bar(phase_pct, "#2fa36b")
            + table
            + "</div>"
        ))

In [ ]:
try:
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
except Exception as error:
    print("WARNING: could not download pretrained weights (turn Internet on). Training from scratch:", error)
    model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)  # one logit: probability of 'real'
model = model.to(device).to(memory_format=torch.channels_last)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)


def run_epoch(loader, dash, train):
    model.train(train)
    dash.phase, dash.in_phase, dash.phase_total = "Training" if train else "Validating", 0, len(loader.dataset)
    total_loss = correct = count = 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            labels = labels.float().to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(images).squeeze(1)
            loss = criterion(logits.float(), labels)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            n = len(labels)
            total_loss += loss.item() * n
            correct += ((logits > 0).float() == labels).sum().item()
            count += n

            dash.in_phase += n
            dash.done += n
            if train:
                dash.trained += n
            dash.loss, dash.acc = total_loss / count, correct / count
            dash.update()

    dash.update(force=True)
    return total_loss / count, correct / count

In [ ]:
dash = Dashboard(EPOCHS, len(train_loader.dataset), len(val_loader.dataset))
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    dash.epoch = epoch
    started = time.time()
    train_loss, train_acc = run_epoch(train_loader, dash, train=True)
    val_loss, val_acc = run_epoch(val_loader, dash, train=False)

    dash.history.append(dict(
        epoch=epoch, train_loss=train_loss, train_acc=train_acc,
        val_loss=val_loss, val_acc=val_acc, seconds=time.time() - started,
    ))
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({"model": "efficientnet", "state_dict": model.state_dict()}, OUT_DIR / "best.pt")

dash.phase, dash.in_phase, dash.phase_total = "Done", 1, 1
dash.update(force=True)
(OUT_DIR / "history.json").write_text(json.dumps(dash.history, indent=2))
print(f"Best validation accuracy: {best_acc:.4f}  (saved {OUT_DIR / 'best.pt'})")

## Test-set results (best checkpoint)

In [ ]:
model.load_state_dict(torch.load(OUT_DIR / "best.pt", map_location=device)["state_dict"])
model.eval()

scores, labels = [], []
with torch.no_grad():
    for images, y in test_loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(images).squeeze(1)
        scores.append(torch.sigmoid(logits.float()).cpu().numpy())
        labels.append(y.numpy())
scores, labels = np.concatenate(scores), np.concatenate(labels)
preds = (scores > 0.5).astype(int)

print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))
print("ROC AUC:", round(roc_auc_score(labels, scores), 4))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], CLASS_NAMES)
ax.set_yticks([0, 1], CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")
fig.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()